# Lesson 3.8 — Reshaping Data (Pivot/Melt)

**Objectives**
- Convert data between "wide" and "long" formats
- Use `.pivot_table()` to summarize data into a wide table
- Use `.melt()` to unpivot a wide table into long format

See `modules/03-pandas/notes.md` (Lesson 3.8) for the full written explanation.


In [1]:
import pandas as pd
from data_science_course.datasets import load_customers, load_orders, load_products

orders = load_orders().drop_duplicates(subset=["order_id"])
customers = load_customers()
products = load_products()

## Build a long table: orders + tier + category

In [2]:
merged = orders.merge(customers[["customer_id", "membership_tier"]], on="customer_id", how="inner")
merged = merged.merge(products[["product_id", "category"]], on="product_id", how="inner")
merged[["order_id", "membership_tier", "category", "order_total"]].head(3)

,order_id,membership_tier,category,order_total
0,O000080,Bronze,Clothing,72.48
1,O002751,Bronze,Sports & Outdoors,69.18
2,O003297,Bronze,Home & Kitchen,201.03


This is long format: one row per order, with `membership_tier` and `category`
identifying what's being measured.

## `.pivot_table()`: long -> wide

In [3]:
wide = merged.pivot_table(
    index="membership_tier",
    columns="category",
    values="order_total",
    aggfunc="sum",
)
wide.round(0)

category,Beauty,Books,Clothing,Electronics,Home & Kitchen,Sports & Outdoors
membership_tier,,,,,,
Bronze,24699.0,24330.0,49835.0,379978.0,79900.0,128401.0
Gold,7162.0,6575.0,13567.0,115910.0,16687.0,49601.0
Platinum,3691.0,2904.0,6508.0,78220.0,11967.0,18350.0
Silver,19175.0,21406.0,45518.0,350013.0,69232.0,113251.0


Rows = `membership_tier`, columns = `category`, cells = summed `order_total`.
This is `.groupby()` from Lesson 3.6 plus one extra step: group by two columns, then
"unstack" one of them into the column axis instead of a MultiIndex.

In [4]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders["order_date"].dt.to_period("M").astype(str)

monthly = orders.pivot_table(
    index="month", columns="status", values="order_total", aggfunc="sum", fill_value=0
)
monthly.round(0).tail(6)

status,cancelled,completed,returned
month,,,
2025-01,3928.0,69049.0,2602.0
2025-02,3034.0,53511.0,2622.0
2025-03,4351.0,59843.0,10024.0
2025-04,2646.0,63603.0,3423.0
2025-05,95.0,63534.0,4656.0
2025-06,1004.0,74979.0,4677.0


`fill_value=0` fills combinations with no data (e.g. a month with zero cancelled
orders) with 0 instead of `NaN`.

## `.melt()`: wide -> long

In [5]:
long = wide.reset_index().melt(
    id_vars="membership_tier",
    var_name="category",
    value_name="total_revenue",
)
long.head(8)

,membership_tier,category,total_revenue
0,Bronze,Beauty,24698.65
1,Gold,Beauty,7162.46
2,Platinum,Beauty,3691.13
3,Silver,Beauty,19174.53
4,Bronze,Books,24329.55
5,Gold,Books,6575.23
6,Platinum,Books,2904.32
7,Silver,Books,21405.53


In [6]:
print(wide.shape, "->", long.shape)  # 4 rows x 6 category-columns -> 24 rows x 3 columns

(4, 6) -> (24, 3)


`wide` was 4 tiers x 6 categories; `id_vars="membership_tier"` keeps that column
fixed and melts the 6 category columns down into (category, total_revenue) pairs,
giving 4 x 6 = 24 rows. This is a round trip back to what
`merged.groupby(["membership_tier", "category"])["order_total"].sum()` would give you
directly -- the reason to pivot at all is that the *wide* shape in the middle is what
a human reads easily in a report, while *long* is what pandas/plotting/ML tools want
as input.

In [7]:
direct_long = (
    merged.groupby(["membership_tier", "category"])["order_total"]
    .sum()
    .reset_index()
    .rename(columns={"order_total": "total_revenue"})
)
# Sort both the same way before comparing, since groupby and melt don't
# necessarily produce rows in the same order.
a = long.sort_values(["membership_tier", "category"]).reset_index(drop=True).round(2)
b = direct_long.sort_values(["membership_tier", "category"]).reset_index(drop=True).round(2)
a.equals(b)

True

## Try it yourself

1. Build a wide pivot table of average (not summed) `order_total` by `status`
   (rows) and `payment_method` (columns).
2. Melt the pivot table from #1 back into long format with columns `status`,
   `payment_method`, `avg_order_total`.
3. Build a wide pivot table showing order **count** (not revenue) by
   `membership_tier` and `category`, using `aggfunc="count"` on `order_id`.
4. Using the `monthly` pivot table built above, compute which month had the highest
   total `completed` revenue.


In [8]:
# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO


### Solution

In [9]:
# 1.
avg_wide = orders.pivot_table(
    index="status", columns="payment_method", values="order_total", aggfunc="mean"
)
print(avg_wide.round(2))

# 2.
avg_long = avg_wide.reset_index().melt(
    id_vars="status", var_name="payment_method", value_name="avg_order_total"
)
print(avg_long.head())

# 3.
count_wide = merged.pivot_table(
    index="membership_tier", columns="category", values="order_id", aggfunc="count"
)
print(count_wide)

# 4.
print(monthly["completed"].idxmax(), monthly["completed"].max())

payment_method  bank_transfer  credit_card  gift_card  paypal
status                                                       
cancelled              349.49       266.40     310.11  294.86
completed              266.31       271.51     274.00  279.29
returned               239.90       315.08     227.99  251.73
      status payment_method  avg_order_total
0  cancelled  bank_transfer       349.492642
1  completed  bank_transfer       266.314876
2   returned  bank_transfer       239.895521
3  cancelled    credit_card       266.396296
4  completed    credit_card       271.514438
category         Beauty  Books  Clothing  Electronics  Home & Kitchen  \
membership_tier                                                         
Bronze              400    595       389          535             248   
Gold                135    154       116          155              61   
Platinum             49     78        61           89              43   
Silver              321    520       356          494  